# AutoEncoders_Memoire_MABANZA — Analyse des résultats (sans ré-entrainement)
Ce notebook charge les artefacts entraînés (AE, DAE, SAE) et calcule des indicateurs quantitatifs pour alimenter la rédaction des sections *Résultats obtenus* et *Discussion*.

Objectifs principaux
- Reproduire les erreurs de reconstruction sur validation, test et sur l'ensemble du dataset
- Calculer des statistiques numériques robustes (moyenne, médiane, écart-type, quantiles, etc.)
- Quantifier des proportions utiles pour la rédaction (p% sous seuil, p% quasi-nulles, proximité du seuil)
- Estimer les variables les plus associées aux fluctuations (corrélations, contributions par feature)
- Exporter des tableaux CSV utilisables dans LaTeX


In [1]:
# =========================
# 0) Configuration des chemins 
# =========================
CSV_PATH = "dataset_autoencoder_all.csv"

ART_AE  = "artifacts_ae"
ART_DAE = "artifacts_dae"
ART_SAE = "artifacts_sae"

OUT_DIR = "analysis_export"   # sorties (tableaux, éventuellement figures)
FIG_DIR = "figures_export"    # si tu veux réutiliser le dossier d'images existant

import os
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

# Vérifications rapides (messages clairs si un fichier manque)
def _assert_exists(path: str, kind: str = "fichier"):
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"{kind} introuvable: {path}\n\n"
            "Actions:\n"
            "- Vérifie que tu exécutes ce notebook dans le même dossier que tes artefacts et le CSV\n"
            "- Ou adapte les variables CSV_PATH / ART_AE / ART_DAE / ART_SAE\n"
        )

_assert_exists(CSV_PATH, kind="CSV")
_assert_exists(ART_AE, kind="dossier")
_assert_exists(ART_DAE, kind="dossier")
_assert_exists(ART_SAE, kind="dossier")

print("OK — chemins validés")

OK — chemins validés


In [2]:
# =========================
# 1) Imports & device
# =========================
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

import joblib
from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE, "| torch:", torch.__version__)

DEVICE: cpu | torch: 2.8.0+cu128


## Chargement des artefacts
On charge le scaler, les métadonnées (colonnes, seuils) et les poids des modèles.

In [3]:
# =========================
# 2) Chargement des artefacts (scaler, métadonnées, seuils)
# =========================
meta_ae  = joblib.load(os.path.join(ART_AE,  "meta.joblib"))
meta_dae = joblib.load(os.path.join(ART_DAE, "meta_dae.joblib"))
meta_sae = joblib.load(os.path.join(ART_SAE, "meta_sae.joblib"))

scaler = joblib.load(os.path.join(ART_AE, "scaler.joblib"))

threshold_ae  = float(meta_ae["threshold_value"])
threshold_dae = float(meta_dae["threshold_value"])
threshold_sae = float(meta_sae["threshold_value"])

feature_cols  = list(meta_ae["feature_cols"])
entropy_cols  = list(meta_ae.get("entropy_cols", []))
log_cols      = list(meta_ae.get("log_cols", []))

print("Nb features:", len(feature_cols))
print("Seuils:", "AE=", threshold_ae, "| DAE=", threshold_dae, "| SAE=", threshold_sae)
print("Log cols:", len(log_cols), "| Entropy cols:", len(entropy_cols))

Nb features: 36
Seuils: AE= 4.389127254486084 | DAE= 4.146695613861084 | SAE= 4.262524604797363
Log cols: 19 | Entropy cols: 5


## Chargement et prétraitement du dataset
Le prétraitement est aligné avec celui utilisé à l'entraînement, via les métadonnées sauvegardées.

In [4]:
# =========================
# 3) Chargement du dataset + prétraitement identique à l'entraînement
# =========================
df = pd.read_csv(CSV_PATH)
print("Dataset shape:", df.shape)

# temps conservé pour analyses temporelles
if "window_start" in df.columns:
    df["window_start"] = pd.to_datetime(df["window_start"], errors="coerce")
else:
    df["window_start"] = pd.NaT

# Vérification colonnes attendues
missing = [c for c in feature_cols if c not in df.columns]
if missing:
    raise ValueError(f"Colonnes manquantes dans le CSV (exemples): {missing[:10]} (total {len(missing)})")

# Construction X_raw dans le même ordre que l'entraînement
X_raw = df[feature_cols].copy()
X_raw = X_raw.replace([np.inf, -np.inf], np.nan).fillna(0.0)

# Règles de prétraitement
X = X_raw.copy()

# a) clipping des entropies
for c in entropy_cols:
    X[c] = X[c].clip(lower=0.0)

# b) log1p sur les colonnes listées
for c in log_cols:
    X[c] = np.log1p(X[c].astype(np.float64))

X = X.replace([np.inf, -np.inf], np.nan).fillna(0.0)

# scaling
X_all_s = scaler.transform(X).astype(np.float32)
print("X_all_s:", X_all_s.shape)

Dataset shape: (13641, 37)
X_all_s: (13641, 36)


## Split validation / test
On reproduit un split identique au notebook d'export des figures afin de pouvoir comparer les distributions.

In [5]:
# =========================
# 4) Split train/val/test (identique au notebook d'export)
# =========================
X_train, X_tmp = train_test_split(X, test_size=0.30, random_state=SEED, shuffle=True)
X_val, X_test  = train_test_split(X_tmp, test_size=0.50, random_state=SEED, shuffle=True)

X_train_s = scaler.transform(X_train).astype(np.float32)
X_val_s   = scaler.transform(X_val).astype(np.float32)
X_test_s  = scaler.transform(X_test).astype(np.float32)

print("Train/Val/Test:", X_train_s.shape, X_val_s.shape, X_test_s.shape)

BATCH_SIZE = 256
train_loader = DataLoader(TensorDataset(torch.from_numpy(X_train_s)), batch_size=BATCH_SIZE, shuffle=False)
val_loader   = DataLoader(TensorDataset(torch.from_numpy(X_val_s)),   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(TensorDataset(torch.from_numpy(X_test_s)),  batch_size=BATCH_SIZE, shuffle=False)

Train/Val/Test: (9548, 36) (2046, 36) (2047, 36)


## Définition des architectures et chargement des poids
On ré-instancie les modèles et on charge les `state_dict` sauvegardés (aucun ré-entrainement).

In [6]:
# =========================
# 5) Architectures AE / DAE / SAE (héritage) + chargement
# =========================
def add_gaussian_noise(x: torch.Tensor, std: float) -> torch.Tensor:
    if std <= 0:
        return x
    return x + torch.randn_like(x) * std

class MLP_AE(nn.Module):
    def __init__(self, in_dim: int, latent_dim: int = 8, dropout: float = 0.1):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(in_dim, 24),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(24, 16),
            nn.ReLU(),
            nn.Linear(16, latent_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 16),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(16, 24),
            nn.ReLU(),
            nn.Linear(24, in_dim),
        )

    def preprocess(self, x: torch.Tensor, train: bool) -> torch.Tensor:
        return x

    def extra_loss(self, z: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
        return torch.tensor(0.0, device=x.device)

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        return self.encoder(x)

    def decode(self, z: torch.Tensor) -> torch.Tensor:
        return self.decoder(z)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x_in = self.preprocess(x, train=self.training)
        z = self.encode(x_in)
        return self.decode(z)

class MLP_DAE(MLP_AE):
    def __init__(self, in_dim: int, latent_dim: int = 8, dropout: float = 0.1, noise_std: float = 0.05):
        super().__init__(in_dim=in_dim, latent_dim=latent_dim, dropout=dropout)
        self.noise_std = noise_std

    def preprocess(self, x: torch.Tensor, train: bool) -> torch.Tensor:
        # bruit seulement en entrainement
        return add_gaussian_noise(x, self.noise_std) if train else x

class MLP_SAE(MLP_AE):
    def __init__(self, in_dim: int, latent_dim: int = 8, dropout: float = 0.1, l1_lambda: float = 1e-3):
        super().__init__(in_dim=in_dim, latent_dim=latent_dim, dropout=dropout)
        self.l1_lambda = l1_lambda

    def extra_loss(self, z: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
        return self.l1_lambda * torch.mean(torch.abs(z))

in_dim = X_train_s.shape[1]
LATENT_DIM = int(meta_ae.get("latent_dim", 8))
DROPOUT = float(meta_ae.get("dropout", 0.1))
NOISE_STD = float(meta_dae.get("noise_std", 0.05))
L1_LAMBDA = float(meta_sae.get("l1_lambda", 1e-3))

model_ae  = MLP_AE(in_dim=in_dim, latent_dim=LATENT_DIM, dropout=DROPOUT).to(DEVICE)
model_dae = MLP_DAE(in_dim=in_dim, latent_dim=LATENT_DIM, dropout=DROPOUT, noise_std=NOISE_STD).to(DEVICE)
model_sae = MLP_SAE(in_dim=in_dim, latent_dim=LATENT_DIM, dropout=DROPOUT, l1_lambda=L1_LAMBDA).to(DEVICE)

model_ae.load_state_dict(torch.load(os.path.join(ART_AE,  "mlp_ae_lat8.pt"),  map_location=DEVICE))
model_dae.load_state_dict(torch.load(os.path.join(ART_DAE, "mlp_dae_lat8.pt"), map_location=DEVICE))
model_sae.load_state_dict(torch.load(os.path.join(ART_SAE, "mlp_sae_lat8.pt"), map_location=DEVICE))

model_ae.eval(); model_dae.eval(); model_sae.eval()
print("OK — modèles chargés")

OK — modèles chargés


## Calcul des erreurs de reconstruction
On calcule l'erreur MSE par fenêtre (moyennée sur les features) pour validation, test et dataset complet.

In [7]:
# =========================
# 6) Utilitaires : erreurs de reconstruction
# =========================
@torch.no_grad()
def reconstruction_errors(model, loader):
    model.eval()
    errs = []
    for (xb,) in loader:
        xb = xb.to(DEVICE)
        x_hat = model(xb)
        e = torch.mean((x_hat - xb) ** 2, dim=1)  # MSE par fenêtre
        errs.append(e.detach().cpu().numpy())
    return np.concatenate(errs, axis=0)

@torch.no_grad()
def reconstruction_errors_array(model, Xs: np.ndarray, batch_size: int = 1024):
    model.eval()
    errs = []
    for i in range(0, Xs.shape[0], batch_size):
        xb = torch.from_numpy(Xs[i:i+batch_size]).to(DEVICE)
        x_hat = model(xb)
        e = torch.mean((x_hat - xb) ** 2, dim=1)
        errs.append(e.detach().cpu().numpy())
    return np.concatenate(errs, axis=0)

val_err_ae  = reconstruction_errors(model_ae,  val_loader)
test_err_ae = reconstruction_errors(model_ae, test_loader)
all_err_ae  = reconstruction_errors_array(model_ae, X_all_s)

val_err_dae  = reconstruction_errors(model_dae,  val_loader)
test_err_dae = reconstruction_errors(model_dae, test_loader)
all_err_dae  = reconstruction_errors_array(model_dae, X_all_s)

val_err_sae  = reconstruction_errors(model_sae,  val_loader)
test_err_sae = reconstruction_errors(model_sae, test_loader)
all_err_sae  = reconstruction_errors_array(model_sae, X_all_s)

print("AE  val/test/all:", val_err_ae.shape, test_err_ae.shape, all_err_ae.shape)
print("DAE val/test/all:", val_err_dae.shape, test_err_dae.shape, all_err_dae.shape)
print("SAE val/test/all:", val_err_sae.shape, test_err_sae.shape, all_err_sae.shape)

AE  val/test/all: (2046,) (2047,) (13641,)
DAE val/test/all: (2046,) (2047,) (13641,)
SAE val/test/all: (2046,) (2047,) (13641,)


## Statistiques numériques utiles pour la rédaction
On produit des indicateurs directement exploitables dans le chapitre de résultats.

In [8]:
# =========================
# 7) Statistiques (résumé) + proportions utiles
# =========================
def summarize_errors(name: str, errs: np.ndarray, threshold: float, eps_list=(0.0, 1e-6, 1e-4, 1e-3, 1e-2)):
    s = {}
    s["model"] = name
    s["n"] = int(errs.size)
    s["mean"] = float(np.mean(errs))
    s["std"] = float(np.std(errs))
    s["median"] = float(np.median(errs))
    s["min"] = float(np.min(errs))
    s["max"] = float(np.max(errs))
    for q in [0.90, 0.95, 0.99, 0.995, 0.999]:
        s[f"q{int(q*1000):04d}"] = float(np.quantile(errs, q))
    s["threshold"] = float(threshold)
    s["pct_below_threshold"] = float(np.mean(errs <= threshold) * 100.0)
    s["pct_above_threshold"] = float(np.mean(errs > threshold) * 100.0)
    # proximité du seuil
    for m in [0.05, 0.10, 0.20]:
        lo = (1.0 - m) * threshold
        hi = (1.0 + m) * threshold
        s[f"pct_within_{int(m*100)}pct_threshold"] = float(np.mean((errs >= lo) & (errs <= hi)) * 100.0)
        s[f"pct_within_{int(m*100)}pct_below_threshold"] = float(np.mean((errs >= lo) & (errs <= threshold)) * 100.0)
    # quasi-nulles
    for eps in eps_list:
        key = "pct_eq0" if eps == 0.0 else f"pct_le_{eps:g}"
        s[key] = float(np.mean(errs <= eps) * 100.0)
    return s

summary_rows = []
# on résume sur l'ensemble du dataset (utile pour rédaction) + val/test si besoin
summary_rows.append(summarize_errors("AE_all",  all_err_ae,  threshold_ae))
summary_rows.append(summarize_errors("DAE_all", all_err_dae, threshold_dae))
summary_rows.append(summarize_errors("SAE_all", all_err_sae, threshold_sae))

summary_df = pd.DataFrame(summary_rows)
summary_df

,model,n,mean,std,median,min,max,q0900,q0950,q0990,...,pct_within_5pct_below_threshold,pct_within_10pct_threshold,pct_within_10pct_below_threshold,pct_within_20pct_threshold,pct_within_20pct_below_threshold,pct_eq0,pct_le_1e-06,pct_le_0.0001,pct_le_0.001,pct_le_0.01
0,AE_all,13641,0.159493,0.682482,0.044186,0.002642,36.209274,0.294009,0.527105,1.926611,...,0.036654,0.117293,0.065978,0.219925,0.124624,0.0,0.0,0.0,0.0,12.301151
1,DAE_all,13641,0.147523,0.533492,0.046760,0.001508,18.718731,0.300827,0.508797,1.499843,...,0.029323,0.051316,0.029323,0.102632,0.065978,0.0,0.0,0.0,0.0,12.777656
2,SAE_all,13641,0.143249,0.546091,0.044857,0.002053,18.586409,0.273439,0.489875,1.499421,...,0.029323,0.095301,0.043985,0.219925,0.124624,0.0,0.0,0.0,0.0,12.007917


In [9]:
# Export CSV pour insertion dans le mémoire (tableau LaTeX via pandas si besoin)
summary_csv = os.path.join(OUT_DIR, "recon_error_summary_all.csv")
summary_df.to_csv(summary_csv, index=False)
print("Saved:", summary_csv)

Saved: analysis_export/recon_error_summary_all.csv


## Comptages d'anomalies et ratios
On calcule le nombre d'anomalies (erreur > seuil) sur dataset complet, et sur val/test.

In [10]:
def anomaly_counts(name: str, errs: np.ndarray, threshold: float):
    n = int(errs.size)
    k = int(np.sum(errs > threshold))
    return {"model": name, "n": n, "anomalies": k, "anomaly_rate_pct": (k / max(n,1))*100.0}

counts = pd.DataFrame([
    anomaly_counts("AE_val",  val_err_ae,  threshold_ae),
    anomaly_counts("AE_test", test_err_ae, threshold_ae),
    anomaly_counts("AE_all",  all_err_ae,  threshold_ae),
    anomaly_counts("DAE_val",  val_err_dae,  threshold_dae),
    anomaly_counts("DAE_test", test_err_dae, threshold_dae),
    anomaly_counts("DAE_all",  all_err_dae,  threshold_dae),
    anomaly_counts("SAE_val",  val_err_sae,  threshold_sae),
    anomaly_counts("SAE_test", test_err_sae, threshold_sae),
    anomaly_counts("SAE_all",  all_err_sae,  threshold_sae),
])
counts

,model,n,anomalies,anomaly_rate_pct
0,AE_val,2046,11,0.537634
1,AE_test,2047,6,0.293112
2,AE_all,13641,47,0.344550
3,DAE_val,2046,11,0.537634
4,DAE_test,2047,5,0.244260
5,DAE_all,13641,35,0.256579
6,SAE_val,2046,11,0.537634
7,SAE_test,2047,5,0.244260
8,SAE_all,13641,41,0.300564


In [11]:
counts_csv = os.path.join(OUT_DIR, "anomaly_counts_rates.csv")
counts.to_csv(counts_csv, index=False)
print("Saved:", counts_csv)

Saved: analysis_export/anomaly_counts_rates.csv


## Variables associées aux fluctuations
Deux angles complémentaires :
1. Corrélation entre chaque feature (valeur) et l'erreur de reconstruction
2. Contribution par feature via l'erreur quadratique moyenne par dimension

Ces résultats alimentent directement la discussion sur les sources probables de fluctuations.

In [12]:
# =========================
# 8) Corrélations feature ↔ erreur (Spearman, robuste aux non-linéarités monotones)
# =========================
def spearman_feature_correlations(X_df: pd.DataFrame, err: np.ndarray):
    rows=[]
    y = err.astype(np.float64)
    for c in X_df.columns:
        x = X_df[c].to_numpy(dtype=np.float64)
        # gestion NaN
        mask = np.isfinite(x) & np.isfinite(y)
        if mask.sum() < 5:
            rho, p = np.nan, np.nan
        else:
            rho, p = spearmanr(x[mask], y[mask])
        rows.append({"feature": c, "spearman_rho": rho, "p_value": p})
    out = pd.DataFrame(rows).sort_values("spearman_rho", key=lambda s: np.abs(s), ascending=False)
    return out

# Corrélations sur tout le dataset, pour chaque modèle
corr_ae  = spearman_feature_correlations(X, all_err_ae)
corr_dae = spearman_feature_correlations(X, all_err_dae)
corr_sae = spearman_feature_correlations(X, all_err_sae)

corr_ae.head(10)

,feature,spearman_rho,p_value
26,net_unique_dst_port,0.530594,0.0
0,cpu_events,0.501203,0.0
4,cpu_unique_pids,0.494721,0.0
30,priv_event_count,0.492875,0.0
5,cpu_unique_comms,0.490120,0.0
22,net_unique_pids,0.476010,0.0
2,cpu_time_mean,-0.469340,0.0
8,exec_unique_binaries,0.452584,0.0
21,net_event_count,0.444279,0.0
7,exec_count,0.432122,0.0


In [13]:
# Export corrélations (top associations)
corr_ae.to_csv(os.path.join(OUT_DIR, "corr_spearman_AE.csv"), index=False)
corr_dae.to_csv(os.path.join(OUT_DIR, "corr_spearman_DAE.csv"), index=False)
corr_sae.to_csv(os.path.join(OUT_DIR, "corr_spearman_SAE.csv"), index=False)
print("Saved correlation CSVs in", OUT_DIR)

Saved correlation CSVs in analysis_export


In [14]:
# =========================
# 9) Contribution par feature (MSE par dimension)
# =========================
@torch.no_grad()
def per_feature_mse(model, Xs: np.ndarray, batch_size: int = 512):
    model.eval()
    mse_sum = None
    n = 0
    for i in range(0, Xs.shape[0], batch_size):
        xb = torch.from_numpy(Xs[i:i+batch_size]).to(DEVICE)
        x_hat = model(xb)
        se = (x_hat - xb) ** 2  # (bs, d)
        se_sum = torch.sum(se, dim=0).detach().cpu().numpy()  # (d,)
        if mse_sum is None:
            mse_sum = se_sum
        else:
            mse_sum += se_sum
        n += xb.shape[0]
    mse_mean = mse_sum / max(n, 1)
    return mse_mean

mse_feat_ae  = per_feature_mse(model_ae,  X_all_s)
mse_feat_dae = per_feature_mse(model_dae, X_all_s)
mse_feat_sae = per_feature_mse(model_sae, X_all_s)

feat_mse_df = pd.DataFrame({
    "feature": feature_cols,
    "mse_AE": mse_feat_ae,
    "mse_DAE": mse_feat_dae,
    "mse_SAE": mse_feat_sae,
})
# top features par modèle
feat_mse_df["rank_AE"]  = feat_mse_df["mse_AE"].rank(ascending=False, method="min")
feat_mse_df["rank_DAE"] = feat_mse_df["mse_DAE"].rank(ascending=False, method="min")
feat_mse_df["rank_SAE"] = feat_mse_df["mse_SAE"].rank(ascending=False, method="min")

feat_mse_df.sort_values("mse_AE", ascending=False).head(15)

,feature,mse_AE,mse_DAE,mse_SAE,rank_AE,rank_DAE,rank_SAE
16,proc_churn,0.931645,0.379659,0.428921,1.0,3.0,1.0
34,priv_suspicious_permission_flag,0.378727,0.437986,0.368513,2.0,1.0,3.0
22,net_unique_pids,0.377939,0.434709,0.372087,3.0,2.0,2.0
5,cpu_unique_comms,0.240343,0.223507,0.187562,4.0,5.0,7.0
3,cpu_time_max,0.235532,0.181872,0.206983,5.0,9.0,4.0
19,proc_unique_commands,0.229540,0.147967,0.161683,6.0,14.0,11.0
6,cpu_entropy_comm,0.224419,0.227044,0.155302,7.0,4.0,13.0
1,cpu_time_sum,0.186672,0.182124,0.198403,8.0,8.0,6.0
4,cpu_unique_pids,0.176191,0.194207,0.175691,9.0,7.0,10.0
14,fork_count,0.161172,0.139902,0.152543,10.0,16.0,15.0


In [15]:
feat_mse_csv = os.path.join(OUT_DIR, "per_feature_mse.csv")
feat_mse_df.sort_values("mse_AE", ascending=False).to_csv(feat_mse_csv, index=False)
print("Saved:", feat_mse_csv)

Saved: analysis_export/per_feature_mse.csv


## Synthèse automatique pour la rédaction
Cette cellule génère un texte court (copiable) contenant les chiffres clés.

In [16]:
def format_key_points(summary_df: pd.DataFrame, counts_df: pd.DataFrame):
    lines=[]
    for base in ["AE", "DAE", "SAE"]:
        s = summary_df[summary_df["model"]==f"{base}_all"].iloc[0].to_dict()
        c = counts_df[counts_df["model"]==f"{base}_all"].iloc[0].to_dict()
        lines.append(f"{base} : moyenne={s['mean']:.4f}, médiane={s['median']:.4f}, écart-type={s['std']:.4f}, min={s['min']:.4f}, max={s['max']:.4f}.")
        lines.append(f"{base} : seuil={s['threshold']:.6f}, taux d'anomalies (dataset complet)={c['anomaly_rate_pct']:.2f} % (n={c['n']}, anomalies={c['anomalies']}).")
        lines.append(f"{base} : proportion d'erreurs <= 1e-3 : {s['pct_le_0.001']:.2f} % ; <= 1e-2 : {s['pct_le_0.01']:.2f} %.")
        lines.append(f"{base} : proportion dans [0.9*seuil, seuil] : {s['pct_within_10pct_below_threshold']:.2f} % (zone proche du seuil).")
        lines.append(" ")
    return "\n".join(lines)

print(format_key_points(summary_df, counts))

AE : moyenne=0.1595, médiane=0.0442, écart-type=0.6825, min=0.0026, max=36.2093.
AE : seuil=4.389127, taux d'anomalies (dataset complet)=0.34 % (n=13641, anomalies=47).
AE : proportion d'erreurs <= 1e-3 : 0.00 % ; <= 1e-2 : 12.30 %.
AE : proportion dans [0.9*seuil, seuil] : 0.07 % (zone proche du seuil).
 
DAE : moyenne=0.1475, médiane=0.0468, écart-type=0.5335, min=0.0015, max=18.7187.
DAE : seuil=4.146696, taux d'anomalies (dataset complet)=0.26 % (n=13641, anomalies=35).
DAE : proportion d'erreurs <= 1e-3 : 0.00 % ; <= 1e-2 : 12.78 %.
DAE : proportion dans [0.9*seuil, seuil] : 0.03 % (zone proche du seuil).
 
SAE : moyenne=0.1432, médiane=0.0449, écart-type=0.5461, min=0.0021, max=18.5864.
SAE : seuil=4.262525, taux d'anomalies (dataset complet)=0.30 % (n=13641, anomalies=41).
SAE : proportion d'erreurs <= 1e-3 : 0.00 % ; <= 1e-2 : 12.01 %.
SAE : proportion dans [0.9*seuil, seuil] : 0.04 % (zone proche du seuil).
 


## Notes d'interprétation
Pour la discussion dans le mémoire :
- Les features avec forte MSE moyenne sont celles où la reconstruction est la moins fidèle (sources possibles de dispersion).
- Les corrélations fortes (en valeur absolue) indiquent des variables dont les variations sont les plus associées aux fluctuations de l'erreur.
- Si la proportion de fenêtres proches du seuil est faible, cela suggère une séparation nette et une meilleure robustesse vis-à-vis du choix du seuil.


## Analyses manuelles des features demontrant que les fenêtres anormales ont des features qui se demarquent

In [17]:
# =========================
# 10) Extraction des fenêtres anormales + variables explicatives
# =========================
import numpy as np
import pandas as pd
import torch

@torch.no_grad()
def reconstruct_and_feature_errors(model, Xs: np.ndarray, batch_size: int = 512):
    """
    Retourne pour chaque fenêtre :
    - la reconstruction en espace scalé
    - l'erreur globale MSE par fenêtre
    - l'erreur quadratique par feature
    """
    model.eval()
    recon_chunks = []
    se_chunks = []
    err_chunks = []

    for i in range(0, Xs.shape[0], batch_size):
        xb = torch.from_numpy(Xs[i:i+batch_size]).to(DEVICE)
        x_hat = model(xb)

        se = (x_hat - xb) ** 2                    # (bs, d)
        err = torch.mean(se, dim=1)              # (bs,)

        recon_chunks.append(x_hat.detach().cpu().numpy())
        se_chunks.append(se.detach().cpu().numpy())
        err_chunks.append(err.detach().cpu().numpy())

    Xhat_s = np.vstack(recon_chunks)
    SE_feat = np.vstack(se_chunks)
    ERR = np.concatenate(err_chunks)

    return Xhat_s, SE_feat, ERR


def extract_anomalous_windows(
    model_name: str,
    model,
    threshold: float,
    X_scaled: np.ndarray,
    X_preprocessed: pd.DataFrame,
    X_raw: pd.DataFrame,
    timestamps: pd.Series,
    feature_cols: list,
    scaler,
    top_n: int = 20,
    top_k_features: int = 5
):
    """
    Extrait les fenêtres anormales d'un modèle et donne les top variables contributives.
    - X_scaled      : données réellement injectées au modèle
    - X_preprocessed: données après prétraitement avant scaling
    - X_raw         : données brutes d'origine
    """
    Xhat_s, SE_feat, ERR = reconstruct_and_feature_errors(model, X_scaled)

    # reconstruction inverse du scaling -> espace prétraité
    Xhat_pre = scaler.inverse_transform(Xhat_s)

    # masque anomalies
    mask_anom = ERR > threshold
    idx_all = np.where(mask_anom)[0]

    if len(idx_all) == 0:
        print(f"Aucune anomalie détectée pour {model_name}.")
        return pd.DataFrame(), pd.DataFrame()

    # tri décroissant par erreur
    idx_sorted = idx_all[np.argsort(ERR[idx_all])[::-1]]
    idx_top = idx_sorted[:top_n]

    rows_summary = []
    rows_detail = []

    for rank_pos, idx in enumerate(idx_top, start=1):
        row = {
            "model": model_name,
            "rank": rank_pos,
            "window_index": int(idx),
            "window_start": timestamps.iloc[idx] if idx < len(timestamps) else pd.NaT,
            "reconstruction_error": float(ERR[idx]),
            "threshold": float(threshold),
            "excess_ratio": float(ERR[idx] / threshold) if threshold > 0 else np.nan,
        }

        # top features contributives
        feat_se = SE_feat[idx]
        top_idx = np.argsort(feat_se)[::-1][:top_k_features]

        for j, feat_idx in enumerate(top_idx, start=1):
            feat = feature_cols[feat_idx]

            raw_value = X_raw.iloc[idx][feat]
            pre_value = X_preprocessed.iloc[idx][feat]
            recon_pre_value = Xhat_pre[idx, feat_idx]
            sq_err = feat_se[feat_idx]

            row[f"top{j}_feature"] = feat
            row[f"top{j}_sq_error"] = float(sq_err)
            row[f"top{j}_raw_value"] = float(raw_value) if pd.notna(raw_value) else np.nan
            row[f"top{j}_preprocessed_value"] = float(pre_value) if pd.notna(pre_value) else np.nan
            row[f"top{j}_reconstructed_value"] = float(recon_pre_value)

            rows_detail.append({
                "model": model_name,
                "rank": rank_pos,
                "window_index": int(idx),
                "window_start": timestamps.iloc[idx] if idx < len(timestamps) else pd.NaT,
                "feature": feat,
                "raw_value": float(raw_value) if pd.notna(raw_value) else np.nan,
                "preprocessed_value": float(pre_value) if pd.notna(pre_value) else np.nan,
                "reconstructed_value": float(recon_pre_value),
                "sq_error_feature": float(sq_err),
                "abs_gap_preprocessed": float(abs(pre_value - recon_pre_value)),
            })

        rows_summary.append(row)

    df_summary = pd.DataFrame(rows_summary)
    df_detail = pd.DataFrame(rows_detail).sort_values(
        ["rank", "sq_error_feature"], ascending=[True, False]
    )

    return df_summary, df_detail


# =========================
# Exécution pour les 3 modèles
# =========================
timestamps = df["window_start"] if "window_start" in df.columns else pd.Series([pd.NaT] * len(X_raw))

anom_ae_summary, anom_ae_detail = extract_anomalous_windows(
    model_name="AE",
    model=model_ae,
    threshold=threshold_ae,
    X_scaled=X_all_s,
    X_preprocessed=X,
    X_raw=X_raw,
    timestamps=timestamps,
    feature_cols=feature_cols,
    scaler=scaler,
    top_n=20,
    top_k_features=5
)

anom_dae_summary, anom_dae_detail = extract_anomalous_windows(
    model_name="DAE",
    model=model_dae,
    threshold=threshold_dae,
    X_scaled=X_all_s,
    X_preprocessed=X,
    X_raw=X_raw,
    timestamps=timestamps,
    feature_cols=feature_cols,
    scaler=scaler,
    top_n=20,
    top_k_features=5
)

anom_sae_summary, anom_sae_detail = extract_anomalous_windows(
    model_name="SAE",
    model=model_sae,
    threshold=threshold_sae,
    X_scaled=X_all_s,
    X_preprocessed=X,
    X_raw=X_raw,
    timestamps=timestamps,
    feature_cols=feature_cols,
    scaler=scaler,
    top_n=20,
    top_k_features=5
)

print("AE anomalies extraites :", anom_ae_summary.shape, anom_ae_detail.shape)
print("DAE anomalies extraites:", anom_dae_summary.shape, anom_dae_detail.shape)
print("SAE anomalies extraites:", anom_sae_summary.shape, anom_sae_detail.shape)

display(anom_ae_summary.head(10))
display(anom_dae_summary.head(10))
display(anom_sae_summary.head(10))

AE anomalies extraites : (20, 32) (100, 10)
DAE anomalies extraites: (20, 32) (100, 10)
SAE anomalies extraites: (20, 32) (100, 10)


,model,rank,window_index,window_start,reconstruction_error,threshold,excess_ratio,top1_feature,top1_sq_error,top1_raw_value,...,top4_feature,top4_sq_error,top4_raw_value,top4_preprocessed_value,top4_reconstructed_value,top5_feature,top5_sq_error,top5_raw_value,top5_preprocessed_value,top5_reconstructed_value
0,AE,1,725,2025-12-02 13:23:06,36.209274,4.389127,8.249766,proc_churn,1138.398560,-1.090000e+02,...,proc_entropy_commands,4.052410,4.390863e+00,4.390863,2.584263,net_entropy_dst_ports,3.859760,3.712323e-01,0.371232,-0.087684
1,AE,2,12111,2025-12-02 13:57:57,25.700485,4.389127,5.855489,proc_churn,629.734985,9.900000e+01,...,exec_shell_count,22.495245,2.200000e+01,3.135494,0.980042,proc_unique_commands,22.145891,2.300000e+01,23.000000,39.263409
2,AE,3,412,2025-12-02 13:17:53,21.623819,4.389127,4.926679,net_unique_pids,554.347168,5.500000e+01,...,cpu_time_max,39.648102,1.844674e+19,44.361420,34.059212,net_suspicious_ports_flag,15.738570,0.000000e+00,0.000000,0.057411
3,AE,4,1783,2025-12-02 14:08:33,16.308969,4.389127,3.715766,cpu_time_sum,225.451843,3.689349e+19,...,exec_unique_binaries,28.590517,3.300000e+01,33.000000,16.124477,exec_shell_count,25.545469,3.000000e+00,1.386294,-0.910647
4,AE,5,13576,2025-12-16 04:10:00,15.544394,4.389127,3.541569,cpu_time_mean,344.659180,4.455735e+16,...,proc_entropy_commands,5.258433,0.000000e+00,0.000000,2.057943,fork_shell,4.779394,0.000000e+00,0.000000,28.760788
5,AE,6,12108,2025-12-02 13:57:54,13.609463,4.389127,3.100722,proc_churn,209.208588,5.800000e+01,...,proc_unique_commands,14.271557,1.700000e+01,17.000000,30.055716,exec_tmp_count,11.750574,0.000000e+00,0.000000,0.042111
6,AE,7,9037,2025-12-02 12:01:41,11.745310,4.389127,2.676001,priv_tmp_mod_count,168.409027,2.000000e+00,...,cpu_time_sum,39.439476,1.844674e+19,44.361420,36.712967,cpu_time_max,33.937424,1.844674e+19,44.361420,34.829979
7,AE,8,8541,2025-12-02 11:53:25,11.435745,4.389127,2.605471,proc_unique_commands,104.281822,3.900000e+01,...,cpu_time_max,51.290989,1.844674e+19,44.361420,32.643791,exec_unique_binaries,25.303406,1.700000e+01,17.000000,1.124197
8,AE,9,4929,2025-12-16 19:18:55,10.559177,4.389127,2.405758,cpu_time_sum,232.908218,0.000000e+00,...,priv_script_mod_count,13.969980,0.000000e+00,0.000000,0.281681,cpu_entropy_comm,13.366261,0.000000e+00,0.000000,2.189483
9,AE,10,12134,2025-12-02 13:58:20,10.487821,4.389127,2.389500,proc_churn,321.232025,-5.300000e+01,...,priv_entropy_filename,7.382072,2.584963e+00,2.584963,0.444005,proc_unique_commands,5.109847,2.400000e+01,24.000000,16.187881


,model,rank,window_index,window_start,reconstruction_error,threshold,excess_ratio,top1_feature,top1_sq_error,top1_raw_value,...,top4_feature,top4_sq_error,top4_raw_value,top4_preprocessed_value,top4_reconstructed_value,top5_feature,top5_sq_error,top5_raw_value,top5_preprocessed_value,top5_reconstructed_value
0,DAE,1,12120,2025-12-02 13:58:06,18.718731,4.146696,4.514132,net_unique_pids,277.565887,5.700000e+01,...,net_unique_dst_port,44.556812,12.000000,12.000000,6.177829,exec_shell_count,22.202868,23.0,3.178054,1.036655
1,DAE,2,412,2025-12-02 13:17:53,16.920809,4.146696,4.080553,net_unique_pids,502.640839,5.500000e+01,...,priv_uid0_count,7.945501,50.000000,3.931826,1.070816,fork_count,6.393040,51.0,3.951244,1.397275
2,DAE,3,1783,2025-12-02 14:08:33,16.750797,4.146696,4.039553,cpu_time_sum,155.219330,3.689349e+19,...,fork_shell,43.362301,3.000000,3.000000,-83.630440,exec_shell_count,23.270018,3.0,1.386294,-0.805962
3,DAE,4,725,2025-12-02 13:23:06,13.387057,4.146696,3.228368,proc_churn,360.918030,-1.090000e+02,...,net_entropy_dst_ports,9.814754,0.371232,0.371232,-0.360568,priv_uid0_count,9.740156,0.0,0.000000,-3.167683
4,DAE,5,8541,2025-12-02 11:53:25,12.698278,4.146696,3.062264,proc_unique_commands,91.777512,3.900000e+01,...,net_entropy_dst_ports,42.616650,0.543564,0.543564,2.068469,net_suspicious_ports_flag,34.702919,0.0,0.000000,0.085250
5,DAE,6,12111,2025-12-02 13:57:57,11.729561,4.146696,2.828652,net_unique_pids,239.847000,4.600000e+01,...,exec_shell_count,26.202139,22.000000,3.135494,0.809218,priv_suspicious_permission_flag,8.444998,0.0,0.000000,0.590492
6,DAE,7,10332,2025-12-02 12:23:16,11.030979,4.146696,2.660185,cpu_time_sum,268.531128,0.000000e+00,...,proc_churn,4.684480,1.000000,1.000000,-6.011275,net_unknown_dst_ip_count,1.624433,0.0,0.000000,2.681909
7,DAE,8,10338,2025-12-02 11:51:02,10.902988,4.146696,2.629320,cpu_time_sum,268.096588,0.000000e+00,...,proc_churn,2.604833,-1.000000,-1.000000,-6.228249,net_unknown_dst_ip_count,1.697290,0.0,0.000000,2.741392
8,DAE,9,4929,2025-12-16 19:18:55,10.895274,4.146696,2.627459,cpu_time_sum,268.060059,0.000000e+00,...,proc_churn,1.797031,-2.000000,-2.000000,-6.342542,net_unknown_dst_ip_count,1.713650,0.0,0.000000,2.754572
9,DAE,10,12745,2025-12-16 03:56:09,10.873406,4.146696,2.622186,proc_churn,223.397827,-4.200000e+01,...,priv_suspicious_permission_flag,30.997503,1.000000,1.000000,-0.131299,exec_unique_binaries,18.836069,18.0,18.000000,31.697502


,model,rank,window_index,window_start,reconstruction_error,threshold,excess_ratio,top1_feature,top1_sq_error,top1_raw_value,...,top4_feature,top4_sq_error,top4_raw_value,top4_preprocessed_value,top4_reconstructed_value,top5_feature,top5_sq_error,top5_raw_value,top5_preprocessed_value,top5_reconstructed_value
0,SAE,1,12120,2025-12-02 13:58:06,18.586409,4.262525,4.360423,net_unique_pids,304.270508,5.700000e+01,...,net_unique_dst_port,38.794777,1.200000e+01,12.00000,6.567315,exec_shell_count,29.629580,2.300000e+01,3.178054,0.704304
1,SAE,2,8541,2025-12-02 11:53:25,17.890169,4.262525,4.197083,cpu_time_sum,219.681381,1.844674e+19,...,exec_unique_binaries,23.664682,1.700000e+01,17.00000,1.646882,exit_count,12.006051,1.630000e+02,5.099866,1.590239
2,SAE,3,412,2025-12-02 13:17:53,16.714933,4.262525,3.921369,net_unique_pids,413.684570,5.500000e+01,...,cpu_time_sum,28.362236,1.844674e+19,44.36142,37.875401,priv_entropy_filename,14.827034,-0.000000e+00,-0.000000,3.034213
3,SAE,4,12111,2025-12-02 13:57:57,14.878202,4.262525,3.490467,proc_churn,238.426071,9.900000e+01,...,priv_entropy_filename,33.791176,1.000000e+00,1.00000,5.580581,net_entropy_dst_ports,18.122917,8.609174e-01,0.860917,1.855331
4,SAE,5,12748,2025-12-16 03:56:12,11.965529,4.262525,2.807146,cpu_time_sum,161.648911,1.844674e+19,...,proc_churn,21.346518,-1.200000e+01,-12.00000,2.966830,exec_unique_binaries,19.919027,1.600000e+01,16.000000,1.914239
5,SAE,6,13576,2025-12-16 04:10:00,11.940243,4.262525,2.801214,cpu_time_mean,294.897675,4.455735e+16,...,priv_script_mod_count,25.702059,0.000000e+00,0.00000,0.382070,cpu_entropy_comm,1.809032,4.776152e+00,4.776152,3.970663
6,SAE,7,725,2025-12-02 13:23:06,11.408293,4.262525,2.676417,proc_churn,313.202179,-1.090000e+02,...,exec_unique_binaries,8.915173,0.000000e+00,0.00000,9.423475,fork_uid0,7.021503,2.000000e+01,20.000000,81.418663
7,SAE,8,12745,2025-12-16 03:56:09,11.357415,4.262525,2.664481,proc_churn,223.076202,-4.200000e+01,...,exec_unique_binaries,22.953480,1.800000e+01,18.00000,33.120651,net_entropy_dst_ports,15.153230,-1.442823e-12,0.000000,0.909297
8,SAE,9,4929,2025-12-16 19:18:55,10.554932,4.262525,2.476216,cpu_time_sum,244.168671,0.000000e+00,...,priv_script_mod_count,9.420536,0.000000e+00,0.00000,0.231311,cpu_entropy_comm,7.133884,0.000000e+00,0.000000,1.599556
9,SAE,10,10332,2025-12-02 12:23:16,10.540410,4.262525,2.472809,cpu_time_sum,245.111237,0.000000e+00,...,priv_script_mod_count,8.779490,0.000000e+00,0.00000,0.223303,cpu_entropy_comm,7.565716,0.000000e+00,0.000000,1.647258


In [18]:
# =========================
# 11) Export CSV
# =========================
anom_ae_summary.to_csv(os.path.join(OUT_DIR, "anomalous_windows_AE_summary.csv"), index=False)
anom_dae_summary.to_csv(os.path.join(OUT_DIR, "anomalous_windows_DAE_summary.csv"), index=False)
anom_sae_summary.to_csv(os.path.join(OUT_DIR, "anomalous_windows_SAE_summary.csv"), index=False)

anom_ae_detail.to_csv(os.path.join(OUT_DIR, "anomalous_windows_AE_detail.csv"), index=False)
anom_dae_detail.to_csv(os.path.join(OUT_DIR, "anomalous_windows_DAE_detail.csv"), index=False)
anom_sae_detail.to_csv(os.path.join(OUT_DIR, "anomalous_windows_SAE_detail.csv"), index=False)

print("CSV exportés dans :", OUT_DIR)

CSV exportés dans : analysis_export


In [19]:
# =========================
# 12) Inspection lisible d'une fenêtre anormale
# =========================
def inspect_window(detail_df: pd.DataFrame, summary_df: pd.DataFrame, rank: int = 1):
    s = summary_df[summary_df["rank"] == rank]
    d = detail_df[detail_df["rank"] == rank].sort_values("sq_error_feature", ascending=False)

    if s.empty:
        print(f"Aucune fenêtre pour rank={rank}")
        return

    print("Résumé de la fenêtre")
    display(s)

    print("\nVariables explicatives")
    display(d[[
        "feature",
        "raw_value",
        "preprocessed_value",
        "reconstructed_value",
        "abs_gap_preprocessed",
        "sq_error_feature"
    ]])

# Exemples
inspect_window(anom_ae_detail, anom_ae_summary, rank=1)
inspect_window(anom_dae_detail, anom_dae_summary, rank=1)
inspect_window(anom_sae_detail, anom_sae_summary, rank=1)

Résumé de la fenêtre


,model,rank,window_index,window_start,reconstruction_error,threshold,excess_ratio,top1_feature,top1_sq_error,top1_raw_value,...,top4_feature,top4_sq_error,top4_raw_value,top4_preprocessed_value,top4_reconstructed_value,top5_feature,top5_sq_error,top5_raw_value,top5_preprocessed_value,top5_reconstructed_value
0,AE,1,725,2025-12-02 13:23:06,36.209274,4.389127,8.249766,proc_churn,1138.39856,-109.0,...,proc_entropy_commands,4.05241,4.390863,4.390863,2.584263,net_entropy_dst_ports,3.85976,0.371232,0.371232,-0.087684



Variables explicatives


,feature,raw_value,preprocessed_value,reconstructed_value,abs_gap_preprocessed,sq_error_feature
0,proc_churn,-109.000000,-109.000000,0.298314,109.298314,1138.398560
1,proc_unique_commands,48.000000,48.000000,8.712313,39.287687,129.235855
2,exit_count,137.000000,4.927254,2.830321,2.096933,4.285951
3,proc_entropy_commands,4.390863,4.390863,2.584263,1.806599,4.052410
4,net_entropy_dst_ports,0.371232,0.371232,-0.087684,0.458916,3.859760


Résumé de la fenêtre


...

,feature,raw_value,preprocessed_value,reconstructed_value,abs_gap_preprocessed,sq_error_feature
0,net_unique_pids,57.0,57.000000,23.727505,33.272495,277.565887
1,exec_tmp_count,1.0,0.693147,0.556734,0.136413,123.306953
2,proc_churn,-24.0,-24.000000,8.855494,32.855494,102.868736
3,net_unique_dst_port,12.0,12.000000,6.177829,5.822171,44.556812
4,exec_shell_count,23.0,3.178054,1.036655,2.141399,22.202868


Résumé de la fenêtre


,model,rank,window_index,window_start,reconstruction_error,threshold,excess_ratio,top1_feature,top1_sq_error,top1_raw_value,...,top4_feature,top4_sq_error,top4_raw_value,top4_preprocessed_value,top4_reconstructed_value,top5_feature,top5_sq_error,top5_raw_value,top5_preprocessed_value,top5_reconstructed_value
0,SAE,1,12120,2025-12-02 13:58:06,18.586409,4.262525,4.360423,net_unique_pids,304.270508,57.0,...,net_unique_dst_port,38.794777,12.0,12.0,6.567315,exec_shell_count,29.62958,23.0,3.178054,0.704304



Variables explicatives


,feature,raw_value,preprocessed_value,reconstructed_value,abs_gap_preprocessed,sq_error_feature
0,net_unique_pids,57.0,57.000000,22.163683,34.836317,304.270508
1,proc_churn,-24.0,-24.000000,8.785862,32.785862,102.433182
2,exec_tmp_count,1.0,0.693147,0.596513,0.096634,61.877815
3,net_unique_dst_port,12.0,12.000000,6.567315,5.432685,38.794777
4,exec_shell_count,23.0,3.178054,0.704304,2.473749,29.629580


## Sur la phase de test uniquement

In [20]:
# =========================
# Extraction des fenêtres anormales - TEST uniquement
# =========================

# 1) Récupérer les indices réels des fenêtres de test
test_idx = X_test.index

# 2) Reconstruire les objets TEST alignés
X_test_preprocessed = X.loc[test_idx].copy()
X_test_raw = X_raw.loc[test_idx].copy()
timestamps_test = df.loc[test_idx, "window_start"].copy()

# 3) Remettre un index propre 0..N-1 pour éviter les décalages dans la fonction
X_test_preprocessed = X_test_preprocessed.reset_index(drop=True)
X_test_raw = X_test_raw.reset_index(drop=True)
timestamps_test = timestamps_test.reset_index(drop=True)

# X_test_s existe déjà dans ton notebook
print("Shapes TEST alignés")
print("X_test_s            :", X_test_s.shape)
print("X_test_preprocessed :", X_test_preprocessed.shape)
print("X_test_raw          :", X_test_raw.shape)
print("timestamps_test     :", timestamps_test.shape)

# 4) Extraction des anomalies sur TEST uniquement
anom_ae_test_summary, anom_ae_test_detail = extract_anomalous_windows(
    model_name="AE",
    model=model_ae,
    threshold=threshold_ae,
    X_scaled=X_test_s,
    X_preprocessed=X_test_preprocessed,
    X_raw=X_test_raw,
    timestamps=timestamps_test,
    feature_cols=feature_cols,
    scaler=scaler,
    top_n=20,
    top_k_features=5
)

anom_dae_test_summary, anom_dae_test_detail = extract_anomalous_windows(
    model_name="DAE",
    model=model_dae,
    threshold=threshold_dae,
    X_scaled=X_test_s,
    X_preprocessed=X_test_preprocessed,
    X_raw=X_test_raw,
    timestamps=timestamps_test,
    feature_cols=feature_cols,
    scaler=scaler,
    top_n=20,
    top_k_features=5
)

anom_sae_test_summary, anom_sae_test_detail = extract_anomalous_windows(
    model_name="SAE",
    model=model_sae,
    threshold=threshold_sae,
    X_scaled=X_test_s,
    X_preprocessed=X_test_preprocessed,
    X_raw=X_test_raw,
    timestamps=timestamps_test,
    feature_cols=feature_cols,
    scaler=scaler,
    top_n=20,
    top_k_features=5
)

# 5) Aperçu
print("AE TEST  :", anom_ae_test_summary.shape, anom_ae_test_detail.shape)
print("DAE TEST :", anom_dae_test_summary.shape, anom_dae_test_detail.shape)
print("SAE TEST :", anom_sae_test_summary.shape, anom_sae_test_detail.shape)

display(anom_ae_test_summary.head(10))
display(anom_dae_test_summary.head(10))
display(anom_sae_test_summary.head(10))

Shapes TEST alignés
X_test_s            : (2047, 36)
X_test_preprocessed : (2047, 36)
X_test_raw          : (2047, 36)
timestamps_test     : (2047,)
AE TEST  : (6, 32) (30, 10)
DAE TEST : (5, 32) (25, 10)
SAE TEST : (5, 32) (25, 10)


,model,rank,window_index,window_start,reconstruction_error,threshold,excess_ratio,top1_feature,top1_sq_error,top1_raw_value,...,top4_feature,top4_sq_error,top4_raw_value,top4_preprocessed_value,top4_reconstructed_value,top5_feature,top5_sq_error,top5_raw_value,top5_preprocessed_value,top5_reconstructed_value
0,AE,1,1512,2025-12-16 03:56:09,9.380851,4.389127,2.137293,proc_churn,221.527664,-4.200000e+01,...,fork_uid0,18.396500,7.570000e+02,757.00000,657.584717,exec_unique_binaries,12.267533,18.000000,18.000000,29.054138
1,AE,2,1041,2025-12-02 13:57:55,8.723599,4.389127,1.987548,net_unique_pids,244.262268,4.500000e+01,...,net_unique_dst_port,6.305502,7.000000e+00,7.00000,4.809780,exec_unique_binaries,5.798905,4.000000,4.000000,11.600099
2,AE,3,945,2025-12-16 03:56:12,8.447197,4.389127,1.924573,cpu_time_mean,76.701134,3.143081e+15,...,cpu_time_max,35.990078,1.844674e+19,44.36142,34.545963,exec_unique_binaries,20.369114,16.000000,16.000000,1.755988
3,AE,4,1784,2025-12-02 13:57:59,6.779136,4.389127,1.544529,net_unique_pids,174.837280,4.200000e+01,...,exec_unique_binaries,8.905360,5.000000e+00,5.00000,14.418288,exec_tmp_count,7.811955,0.000000,0.000000,0.034335
4,AE,5,537,2025-12-16 18:56:05,6.350209,4.389127,1.446804,proc_churn,194.117294,5.200000e+01,...,exec_entropy_filename,5.071384,0.000000e+00,0.00000,2.072258,cpu_unique_comms,2.784419,123.000000,123.000000,101.752190
5,AE,6,168,2025-12-02 13:22:14,5.500380,4.389127,1.253183,net_suspicious_ports_flag,137.443298,1.000000e+00,...,proc_churn,6.182617,-1.200000e+01,-12.00000,-3.945242,net_entropy_dst_ports,3.672307,1.563279,1.563279,1.115646


,model,rank,window_index,window_start,reconstruction_error,threshold,excess_ratio,top1_feature,top1_sq_error,top1_raw_value,...,top4_feature,top4_sq_error,top4_raw_value,top4_preprocessed_value,top4_reconstructed_value,top5_feature,top5_sq_error,top5_raw_value,top5_preprocessed_value,top5_reconstructed_value
0,DAE,1,1512,2025-12-16 03:56:09,10.873406,4.146696,2.622186,proc_churn,223.397827,-4.200000e+01,...,priv_suspicious_permission_flag,30.997503,1.0,1.0,-0.131299,exec_unique_binaries,18.836069,1.800000e+01,18.000000,31.697502
1,DAE,2,1041,2025-12-02 13:57:55,10.114957,4.146696,2.439281,net_unique_pids,301.178802,4.500000e+01,...,exec_unique_binaries,7.213305,4.0,4.0,12.476438,exec_entropy_filename,6.667683,8.059585e-01,0.805958,3.182077
2,DAE,3,1784,2025-12-02 13:57:59,7.835513,4.146696,1.889580,net_unique_pids,231.268829,4.200000e+01,...,proc_unique_commands,5.059860,12.0,12.0,19.773815,fork_uid0,4.308081,5.100000e+01,51.000000,2.890900
3,DAE,4,945,2025-12-16 03:56:12,7.077250,4.146696,1.706720,cpu_time_sum,51.858784,1.844674e+19,...,proc_churn,14.914466,-12.0,-12.0,0.510370,net_entropy_dst_ports,11.777952,0.000000e+00,0.000000,0.801656
4,DAE,5,168,2025-12-02 13:22:14,6.358127,4.146696,1.533300,net_suspicious_ports_flag,47.812225,1.000000e+00,...,fork_shell,14.651378,0.0,0.0,-50.356277,cpu_time_sum,13.791786,3.325613e+09,21.924920,26.447834


,model,rank,window_index,window_start,reconstruction_error,threshold,excess_ratio,top1_feature,top1_sq_error,top1_raw_value,...,top4_feature,top4_sq_error,top4_raw_value,top4_preprocessed_value,top4_reconstructed_value,top5_feature,top5_sq_error,top5_raw_value,top5_preprocessed_value,top5_reconstructed_value
0,SAE,1,945,2025-12-16 03:56:12,11.965529,4.262525,2.807146,cpu_time_sum,161.648911,1.844674e+19,...,proc_churn,21.346518,-12.0,-12.0,2.966830,exec_unique_binaries,19.919027,1.600000e+01,16.000000,1.914239
1,SAE,2,1512,2025-12-16 03:56:09,11.357415,4.262525,2.664481,proc_churn,223.076202,-4.200000e+01,...,exec_unique_binaries,22.953480,18.0,18.0,33.120651,net_entropy_dst_ports,15.153230,-1.442823e-12,0.000000,0.909297
2,SAE,3,1041,2025-12-02 13:57:55,9.698585,4.262525,2.275315,net_unique_pids,259.261658,4.500000e+01,...,exec_unique_binaries,4.838805,4.0,4.0,10.942491,exec_entropy_filename,4.277589,8.059585e-01,0.805958,2.709139
3,SAE,4,1784,2025-12-02 13:57:59,6.717600,4.262525,1.575967,net_unique_pids,182.526962,4.200000e+01,...,proc_unique_commands,5.579604,12.0,12.0,20.163317,proc_churn,4.235247,1.600000e+01,16.000000,22.666618
4,SAE,5,168,2025-12-02 13:22:14,5.891138,4.262525,1.382077,net_suspicious_ports_flag,64.618126,1.000000e+00,...,priv_tmp_mod_count,11.793565,0.0,0.0,0.257379,exec_entropy_filename,10.273058,2.251629e+00,2.251629,5.201007


In [21]:
# =========================
# Export CSV - TEST uniquement
# =========================
anom_ae_test_summary.to_csv(os.path.join(OUT_DIR, "anomalous_windows_AE_test_summary.csv"), index=False)
anom_dae_test_summary.to_csv(os.path.join(OUT_DIR, "anomalous_windows_DAE_test_summary.csv"), index=False)
anom_sae_test_summary.to_csv(os.path.join(OUT_DIR, "anomalous_windows_SAE_test_summary.csv"), index=False)

anom_ae_test_detail.to_csv(os.path.join(OUT_DIR, "anomalous_windows_AE_test_detail.csv"), index=False)
anom_dae_test_detail.to_csv(os.path.join(OUT_DIR, "anomalous_windows_DAE_test_detail.csv"), index=False)
anom_sae_test_detail.to_csv(os.path.join(OUT_DIR, "anomalous_windows_SAE_test_detail.csv"), index=False)

print("CSV TEST exportés dans :", OUT_DIR)

CSV TEST exportés dans : analysis_export
